# Macroeconomic Data Collection Pipeline

---

This notebook collects and compiles macroeconomic time series for the US and UK, to be used as candidate predictor variables in the macroeconomic model for Probability of Default (PD) prediction under IFRS 9.

**Data Sources Used:**
- **FRED** (Federal Reserve Bank of St. Louis): `fredapi` Python library
- **OECD**: Direct API calls via `requests`
- **Yahoo Finance**: `yfinance` Python library

**Output Datasets:**
- `MacroVariables_US_Q.csv` — US quarterly (primary modelling dataset, includes delinquency rate as target variable)
- `MacroVariables_US_M.csv` — US monthly (Stage 1 macro forecasting, spline-interpolated where needed)
- `MacroVariables_UK_Q.csv` — UK quarterly (macro variables only, no target variable)

**Temporal Frequency:** Two datasets per country — quarterly (primary) and monthly (forecasting stage)

**Date Coverage:** All available data for each series, effective analytical window from 1990 onwards

## Variable Reference

---

### Table 1 — US Quarterly Dataset (`MacroVariables_US_Q.csv`)

This is the primary modelling dataset. All variables are indexed to quarter-end dates (e.g. 2024-03-31). The target variable for PD modelling is `us_delinquency_rate`.

| Variable Name | Description | Units | Raw Source Frequency | Transformation to Quarterly | Source | Series ID |
|---|---|---|---|---|---|---|
| `us_real_gdp` | Real GDP, seasonally adjusted annual rate | Billions of chained 2017 USD | Quarterly | Native quarterly, index shifted to quarter-end | FRED | `GDPC1` |
| `us_gdp_yoy_growth` | Year-on-year real GDP growth rate | % | Quarterly | Derived: `pct_change(4) * 100` on `us_real_gdp` | FRED | — |
| `us_cpi` | Consumer Price Index, total, YoY change | % | Quarterly | Native quarterly, used as-is | OECD | — |
| `us_unemployment` | Unemployment rate | % | Quarterly | Native quarterly, used as-is | OECD | — |
| `us_house_price_idx` | Real House Price Index (2015=100) | Index | Quarterly | Native quarterly, used as-is | OECD | — |
| `us_bond_yield_10y` | 10-year Treasury yield | % per annum | Monthly | Last observation of quarter | FRED | `IRLTLT01USM156N` |
| `us_bond_yield_1y` | 1-year Treasury yield | % per annum | Daily | Last observation of quarter | FRED | `DGS1` |
| `us_term_spread` | Term spread (10Y − 1Y Treasury yield) | % | — | Derived: `us_bond_yield_10y` minus `us_bond_yield_1y` | FRED | — |
| `us_reer` | Real Broad Effective Exchange Rate | Index (2020=100) | Monthly | Last observation of quarter | FRED | `RBUSBIS` |
| `us_reer_log_ret` | Log return of REER | — | Monthly | Derived: `ln(us_reer_t / us_reer_{t-1})` on quarterly series | FRED | — |
| `us_oil_price` | WTI Crude Oil spot price | USD per barrel | Daily | Last observation of quarter | FRED | `DCOILWTICO` |
| `us_oil_log_ret` | Log return of WTI oil price | — | Daily | Derived: `ln(us_oil_t / us_oil_{t-1})` on quarterly series | FRED | — |
| `us_credit` | Total credit to non-financial sector, adjusted for breaks | Billions of USD | Quarterly | Native quarterly, index shifted to quarter-end | FRED | `CRDQUSAPABIS` |
| `us_credit_qoq_growth` | Quarter-on-quarter credit growth | % | Quarterly | Derived: `pct_change(1) * 100` on `us_credit` | FRED | — |
| `us_credit_yoy_growth` | Year-on-year credit growth | % | Quarterly | Derived: `pct_change(4) * 100` on `us_credit` | FRED | — |
| `us_industrial_production` | Industrial Production Index | Index (2017=100) | Monthly | Last observation of quarter | FRED | `INDPRO` |
| `us_vix` | CBOE Volatility Index (VIX) | Index points | Daily | Last observation of quarter | FRED | `VIXCLS` |
| `us_vix_log_ret` | Log return of VIX | — | Daily | Derived: `ln(us_vix_t / us_vix_{t-1})` on quarterly series | FRED | — |
| `us_sp500_close` | S&P 500 closing price (last day of quarter) | Index points | Daily | Last observation of quarter | Yahoo Finance | `^GSPC` |
| `us_sp500_log_ret` | Log return of S&P 500 | — | Daily | Derived: `ln(us_sp500_t / us_sp500_{t-1})` on quarterly series | Yahoo Finance | — |
| `us_consumer_confidence` | Consumer Confidence Index (amplitude-adjusted) | Index (long-run avg=100) | Monthly | Last observation of quarter | OECD | — |
| `us_delinquency_rate` | Delinquency rate, all loans, all commercial banks — **target variable** | % | Quarterly | Native quarterly, index shifted to quarter-end | FRED | `DRCCLACBS` |

---

### Table 2 — US Monthly Dataset (`MacroVariables_US_M.csv`)

This dataset is used for Stage 1 of the pipeline (macro variable forecasting via SARIMA/SARIMAX and ML models). All variables are indexed to month-end dates. Variables that are only available at quarterly frequency in their raw form are disaggregated to monthly using cubic spline interpolation — quarter-end anchor values are preserved exactly, and the two intervening months are smoothly estimated.

| Variable Name | Description | Units | Raw Source Frequency | Transformation to Monthly | Source | Series ID |
|---|---|---|---|---|---|---|
| `us_real_gdp` | Real GDP, seasonally adjusted annual rate | Billions of chained 2017 USD | Quarterly | Cubic spline interpolation from quarter-end anchors | FRED | `GDPC1` |
| `us_gdp_yoy_growth` | Year-on-year real GDP growth rate | % | Quarterly | Derived: `pct_change(12) * 100` on monthly `us_real_gdp` | FRED | — |
| `us_cpi` | Consumer Price Index, total, YoY change | % | Quarterly | Cubic spline interpolation from quarter-end anchors | OECD | — |
| `us_unemployment` | Unemployment rate | % | Quarterly | Cubic spline interpolation from quarter-end anchors | OECD | — |
| `us_house_price_idx` | Real House Price Index (2015=100) | Index | Quarterly | Cubic spline interpolation from quarter-end anchors | OECD | — |
| `us_bond_yield_10y` | 10-year Treasury yield | % per annum | Monthly | Last observation of month | FRED | `IRLTLT01USM156N` |
| `us_bond_yield_1y` | 1-year Treasury yield | % per annum | Daily | Last observation of month | FRED | `DGS1` |
| `us_term_spread` | Term spread (10Y − 1Y Treasury yield) | % | — | Derived: `us_bond_yield_10y` minus `us_bond_yield_1y` on monthly series | FRED | — |
| `us_reer` | Real Broad Effective Exchange Rate | Index (2020=100) | Monthly | Last observation of month | FRED | `RBUSBIS` |
| `us_reer_log_ret` | Log return of REER | — | Monthly | Derived: `ln(us_reer_t / us_reer_{t-1})` on monthly series | FRED | — |
| `us_oil_price` | WTI Crude Oil spot price | USD per barrel | Daily | Last observation of month | FRED | `DCOILWTICO` |
| `us_oil_log_ret` | Log return of WTI oil price | — | Daily | Derived: `ln(us_oil_t / us_oil_{t-1})` on monthly series | FRED | — |
| `us_credit` | Total credit to non-financial sector, adjusted for breaks | Billions of USD | Quarterly | Cubic spline interpolation from quarter-end anchors | FRED | `CRDQUSAPABIS` |
| `us_credit_mom_growth` | Month-on-month credit growth | % | Quarterly | Derived: `pct_change(1) * 100` on monthly `us_credit` | FRED | — |
| `us_credit_yoy_growth` | Year-on-year credit growth | % | Quarterly | Derived: `pct_change(12) * 100` on monthly `us_credit` | FRED | — |
| `us_industrial_production` | Industrial Production Index | Index (2017=100) | Monthly | Last observation of month | FRED | `INDPRO` |
| `us_vix` | CBOE Volatility Index (VIX) | Index points | Daily | Last observation of month | FRED | `VIXCLS` |
| `us_vix_log_ret` | Log return of VIX | — | Daily | Derived: `ln(us_vix_t / us_vix_{t-1})` on monthly series | FRED | — |
| `us_sp500_close` | S&P 500 closing price (last day of month) | Index points | Daily | Last observation of month | Yahoo Finance | `^GSPC` |
| `us_sp500_log_ret` | Log return of S&P 500 | — | Daily | Derived: `ln(us_sp500_t / us_sp500_{t-1})` on monthly series | Yahoo Finance | — |
| `us_consumer_confidence` | Consumer Confidence Index (amplitude-adjusted) | Index (long-run avg=100) | Monthly | Native monthly, indexed to month-end | OECD | — |
| `us_delinquency_rate` | Delinquency rate, all loans, all commercial banks — **target variable** | % | Quarterly | Cubic spline interpolation from quarter-end anchors | FRED | `DRCCLACBS` |

---

### Table 3 — UK Quarterly Dataset (`MacroVariables_UK_Q.csv`)

Macro variables only. No target variable is available for the UK at quarterly frequency with sufficient historical coverage. This dataset is retained for macro variable analysis and potential future work.

| Variable Name | Description | Units | Raw Source Frequency | Transformation to Quarterly | Source | Series ID |
|---|---|---|---|---|---|---|
| `uk_real_gdp` | Real GDP, seasonally adjusted | Millions of domestic currency (GBP) | Quarterly | Native quarterly, index shifted to quarter-end | FRED | `NGDPRSAXDCGBQ` |
| `uk_gdp_yoy_growth` | Year-on-year real GDP growth rate | % | Quarterly | Derived: `pct_change(4) * 100` on `uk_real_gdp` | FRED | — |
| `uk_cpi` | Consumer Price Index, total, YoY change | % | Quarterly | Native quarterly, used as-is | OECD | — |
| `uk_unemployment` | Unemployment rate | % | Quarterly | Native quarterly, used as-is | OECD | — |
| `uk_house_price_idx` | Real House Price Index (2015=100) | Index | Quarterly | Native quarterly, used as-is | OECD | — |
| `uk_bond_yield_10y` | Long-term (10-year) government bond yield | % per annum | Monthly | Last observation of quarter | FRED | `IRLTLT01GBM156N` |
| `uk_reer` | Real Broad Effective Exchange Rate | Index (2020=100) | Monthly | Last observation of quarter | FRED | `RBGBBIS` |
| `uk_oil_price` | Brent Crude Oil spot price (Europe) | USD per barrel | Daily | Last observation of quarter | FRED | `DCOILBRENTEU` |
| `uk_credit` | Total credit to non-financial sector, adjusted for breaks | Billions of USD | Quarterly | Native quarterly, index shifted to quarter-end | FRED | `QGBCAMUSDA` |
| `uk_credit_qoq_growth` | Quarter-on-quarter credit growth | % | Quarterly | Derived: `pct_change(1) * 100` on `uk_credit` | FRED | — |
| `uk_credit_yoy_growth` | Year-on-year credit growth | % | Quarterly | Derived: `pct_change(4) * 100` on `uk_credit` | FRED | — |
| `uk_industrial_production` | Industrial Production Index | Index (2012=100) | Monthly | Last observation of quarter | FRED | `IPIUKM` |
| `uk_ftse_close` | FTSE 100 closing price (last day of quarter) | Index points | Daily | Last observation of quarter | Yahoo Finance | `^FTSE` |
| `uk_ftse_qoq_growth` | Quarter-on-quarter FTSE 100 growth | % | Daily | Derived: `pct_change(1) * 100` on `uk_ftse_close` | Yahoo Finance | — |
| `uk_consumer_confidence` | Consumer Confidence Index (amplitude-adjusted) | Index (long-run avg=100) | Monthly | Last observation of quarter | OECD | — |

---

### Notes

- Log returns are computed for financial price variables (VIX, oil, equity index, REER) in both the quarterly and monthly datasets. These are used in the SARIMA/SARIMAX forecasting stage to ensure stationarity. The formula is `ln(P_t / P_{t-1})`.
- Economic flow variables (GDP, credit, industrial production) use YoY or QoQ growth rates rather than log returns, as these series have additive rather than multiplicative dynamics and YoY differencing removes seasonality.
- Rate variables (bond yields, unemployment, term spread, delinquency rate) are kept in levels as they are already stationary or near-stationary and growth rates of rates are not economically meaningful.
- UK Industrial Production (`IPIUKM`) is only available until 2017, limiting its usefulness in any forward-looking model.
- VIX is a US-specific index. No equivalent is collected for the UK.
- The effective analytical window for all datasets is 1990 onwards due to data availability across all variables.

## 1. Install & Import Packages
---

In [24]:
#pip install fredapi yfinance scipy

In [25]:
import pandas as pd
import numpy as np
import requests
import time
import os
from io import StringIO
from pathlib import Path
from scipy.interpolate import CubicSpline
from fredapi import Fred
import yfinance as yf

## 2. API Setup
---

In [26]:
# Load FRED API key from a local text file
# with open("fred_api.txt", "r") as f:
#     fred_api_key = f.read().strip()
    
fred = Fred(api_key='ca04d9a1acca4616dd8c051986a1e29e') # Replace with your actual FRED API key

## 3. Helper Functions
---
Three categories of helpers are defined here:

**Resampling functions** convert high-frequency raw series to the target output frequency by taking the last observation of each period. This preserves the end-of-period value which is the economically relevant observation for quarterly and monthly datasets.

**Interpolation function** disaggregates quarterly series to monthly using cubic spline interpolation (`scipy.interpolate.CubicSpline`). Quarter-end anchor values are preserved exactly. The two intervening months are smoothly estimated from the fitted spline curve.

**OECD fetch functions** handle the OECD SDMX REST API, including rate-limit delays and date parsing for both quarterly (`YYYY-Qn`) and monthly (`YYYY-MM`) period formats.

In [27]:
def resample_to_quarter_end(series):
    """
    Resample a time series to quarterly frequency using the last available
    observation in each quarter, anchored to the last day of the quarter.
    FRED dates observations to the start of the period but the value
    represents the end of that period, so we align to quarter-end.
    Strips timezone info to ensure consistency across sources.
    """
    series.index = pd.to_datetime(series.index).tz_localize(None)
    return series.resample('QE').last()


def resample_to_month_end(series):
    """
    Resample a daily or sub-monthly series to monthly frequency using
    the last available observation in each month, anchored to month-end.
    Strips timezone info to ensure consistency across sources.
    """
    series.index = pd.to_datetime(series.index).tz_localize(None)
    return series.resample('ME').last()


def quarterly_to_monthly_spline(quarterly_series):
    """
    Interpolate a quarterly time series to monthly frequency using cubic
    spline interpolation. Quarter-end anchor values are preserved exactly.
    Intermediate months are smoothly estimated from the spline curve.

    Parameters
    ----------
    quarterly_series : pd.Series
        Series with quarter-end DatetimeIndex. NaNs are dropped before fitting.

    Returns
    -------
    pd.Series
        Monthly series with month-end DatetimeIndex.
    """
    s = quarterly_series.dropna()
    if len(s) < 4:
        raise ValueError(f"Too few observations to fit spline: {quarterly_series.name}")

    # Convert dates to ordinal integers (days since year 1) for spline fitting
    x = np.array([d.toordinal() for d in s.index])
    y = s.values.astype(float)

    cs = CubicSpline(x, y)

    # Build monthly date range anchored to month-end
    monthly_idx = pd.date_range(
        start = s.index.min().to_period('M').to_timestamp('M'),
        end   = s.index.max().to_period('M').to_timestamp('M'),
        freq  = 'ME'
    )

    x_monthly = np.array([d.toordinal() for d in monthly_idx])
    y_monthly  = cs(x_monthly)

    return pd.Series(y_monthly, index=monthly_idx, name=quarterly_series.name)


def fetch_oecd(url, col_name):
    """
    Fetch a single time series from the OECD SDMX REST API.
    Returns a two-column DataFrame: ['date', col_name].
    Adds a 1-second delay between calls to respect OECD rate limits.
    """
    headers = {'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both'}
    time.sleep(1)

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))

        time_col = [c for c in df.columns if c.startswith('TIME_PERIOD')][0]
        df_clean = df[[time_col, 'OBS_VALUE']].copy()
        df_clean.columns = ['date', col_name]
        df_clean[col_name] = pd.to_numeric(df_clean[col_name], errors='coerce')
        return df_clean
    else:
        print(f"WARNING: Could not fetch '{col_name}' - HTTP {response.status_code}")
        return None


def fetch_oecd_native_monthly(url, col_name):
    """
    Fetch a monthly OECD series and return a Series indexed to month-end
    timestamps without resampling to quarterly. Used for the monthly dataset
    where consumer confidence is kept at its native monthly frequency.
    """
    df = fetch_oecd(url, col_name)
    if df is None:
        return None
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    df.index = df.index.to_period('M').to_timestamp('M')
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[col_name]

## 4. FRED Data
---
Monthly and daily FRED series are resampled to quarterly frequency by taking the last observation of each quarter, anchored to the final day of the quarter (e.g. 2025-03-31). Quarterly series are shifted from quarter-start to quarter-end indexing to ensure alignment across all sources.

NOTE: The values of the most recent quarter may be incomplete if the quarter has not yet ended. The current incomplete quarter is dropped at the combine step in Section 7.

Two sub-sections follow — UK series first, then US series.

### 4a. UK — FRED Series

In [28]:
# UK Real GDP (quarterly, seasonally adjusted)
# Source: FRED NGDPRSAXDCGBQ
uk_real_gdp_raw = fred.get_series('NGDPRSAXDCGBQ')
uk_real_gdp_raw.index = uk_real_gdp_raw.index.to_period('Q').to_timestamp('Q')
uk_real_gdp       = uk_real_gdp_raw.copy()
uk_gdp_yoy_growth = uk_real_gdp_raw.pct_change(4) * 100

# UK Long-Term Government Bond Yield (10-year, monthly to quarterly)
# Source: FRED IRLTLT01GBM156N
uk_bond_yield_10y = resample_to_quarter_end(fred.get_series('IRLTLT01GBM156N'))

# UK Real Broad Effective Exchange Rate (monthly to quarterly)
# Source: FRED RBGBBIS
uk_reer = resample_to_quarter_end(fred.get_series('RBGBBIS'))

# UK Brent Crude Oil Price (daily to quarterly)
# Source: FRED DCOILBRENTEU
uk_oil_price = resample_to_quarter_end(fred.get_series('DCOILBRENTEU'))

# UK Total Credit to Non-Financial Sector, Adjusted for Breaks (quarterly)
# Source: FRED QGBCAMUSDA
uk_credit_raw = fred.get_series('QGBCAMUSDA')
uk_credit_raw.index  = uk_credit_raw.index.to_period('Q').to_timestamp('Q')
uk_credit            = uk_credit_raw.copy()
uk_credit_qoq_growth = uk_credit_raw.pct_change(1) * 100
uk_credit_yoy_growth = uk_credit_raw.pct_change(4) * 100

# UK Industrial Production Index (monthly to quarterly)
# Source: FRED IPIUKM
uk_industrial_production = resample_to_quarter_end(fred.get_series('IPIUKM'))

# Compile UK FRED DataFrame
fred_uk = pd.DataFrame({
    'uk_real_gdp':              uk_real_gdp,
    'uk_gdp_yoy_growth':        uk_gdp_yoy_growth,
    'uk_bond_yield_10y':        uk_bond_yield_10y,
    'uk_reer':                  uk_reer,
    'uk_oil_price':             uk_oil_price,
    'uk_credit':                uk_credit,
    'uk_credit_qoq_growth':     uk_credit_qoq_growth,
    'uk_credit_yoy_growth':     uk_credit_yoy_growth,
    'uk_industrial_production': uk_industrial_production,
})

fred_uk.index = pd.to_datetime(fred_uk.index).tz_localize(None)
print(f"UK FRED data: {fred_uk.index.min().date()} to {fred_uk.index.max().date()}, {len(fred_uk)} rows")

UK FRED data: 1920-03-31 to 2026-06-30, 426 rows


In [29]:
fred_uk.head()

,uk_real_gdp,uk_gdp_yoy_growth,uk_bond_yield_10y,uk_reer,uk_oil_price,uk_credit,uk_credit_qoq_growth,uk_credit_yoy_growth,uk_industrial_production
1920-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.69
1920-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.69
1920-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.77
1920-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.89
1921-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.69


In [30]:
fred_uk.tail()

,uk_real_gdp,uk_gdp_yoy_growth,uk_bond_yield_10y,uk_reer,uk_oil_price,uk_credit,uk_credit_qoq_growth,uk_credit_yoy_growth,uk_industrial_production
2025-06-30,705117.0,1.434521,4.5248,113.75,68.15,8916.353,7.579480,10.028417,NaN
2025-09-30,705681.0,1.281518,4.6885,111.89,68.52,8843.121,-0.821322,1.413113,NaN
2025-12-31,706067.0,0.997434,4.4826,111.51,61.35,NaN,NaN,NaN,NaN
2026-03-31,NaN,NaN,4.4324,111.25,126.69,NaN,NaN,NaN,NaN
2026-06-30,NaN,NaN,NaN,NaN,127.61,NaN,NaN,NaN,NaN


### 4b. US — FRED Series

In [31]:
# US Real GDP (quarterly, seasonally adjusted annual rate)
# Source: FRED GDPC1
us_real_gdp_raw = fred.get_series('GDPC1')
us_real_gdp_raw.index = us_real_gdp_raw.index.to_period('Q').to_timestamp('Q')
us_real_gdp       = us_real_gdp_raw.copy()
us_gdp_yoy_growth = us_real_gdp_raw.pct_change(4) * 100

# US Long-Term Government Bond Yield (10-year Treasury, monthly to quarterly)
# Source: FRED IRLTLT01USM156N
us_bond_yield_10y = resample_to_quarter_end(fred.get_series('IRLTLT01USM156N'))

# US Short-Term Government Bond Yield (1-year Treasury, daily to quarterly)
# Source: FRED DGS1
us_bond_yield_1y = resample_to_quarter_end(fred.get_series('DGS1'))

# US Term Spread (10Y - 1Y)
us_term_spread = (us_bond_yield_10y - us_bond_yield_1y).rename('us_term_spread')

# US Real Broad Effective Exchange Rate (monthly to quarterly)
# Source: FRED RBUSBIS
us_reer = resample_to_quarter_end(fred.get_series('RBUSBIS'))

# US WTI Crude Oil Price (daily to quarterly)
# Source: FRED DCOILWTICO
us_oil_price = resample_to_quarter_end(fred.get_series('DCOILWTICO'))

# US Total Credit to Non-Financial Sector, Adjusted for Breaks (quarterly)
# Source: FRED CRDQUSAPABIS
us_credit_raw = fred.get_series('CRDQUSAPABIS')
us_credit_raw.index  = us_credit_raw.index.to_period('Q').to_timestamp('Q')
us_credit            = us_credit_raw.copy()
us_credit_qoq_growth = us_credit_raw.pct_change(1) * 100
us_credit_yoy_growth = us_credit_raw.pct_change(4) * 100

# US Industrial Production: Total Index (monthly to quarterly)
# Source: FRED INDPRO
us_industrial_production = resample_to_quarter_end(fred.get_series('INDPRO'))

# VIX: CBOE Volatility Index (daily to quarterly)
# Source: FRED VIXCLS
us_vix = resample_to_quarter_end(fred.get_series('VIXCLS'))

# US Delinquency Rate, All Loans, All Commercial Banks (quarterly)
# Source: FRED DRCCLACBS — primary target variable for PD modelling
us_delinquency_raw = fred.get_series('DRCCLACBS')
us_delinquency_raw.index = us_delinquency_raw.index.to_period('Q').to_timestamp('Q')
us_delinquency_rate = us_delinquency_raw.copy()

# Log returns for financial price variables
# Formula: ln(P_t / P_{t-1})
# Applied to VIX, oil, and REER — economic flow variables use growth rates instead
us_vix_log_ret  = np.log(us_vix / us_vix.shift(1)).rename('us_vix_log_ret')
us_oil_log_ret  = np.log(us_oil_price / us_oil_price.shift(1)).rename('us_oil_log_ret')
us_reer_log_ret = np.log(us_reer / us_reer.shift(1)).rename('us_reer_log_ret')

# US Consumer Price Index — YoY growth rate (monthly to quarterly)
# Source: FRED CPIAUCSL (Bureau of Labor Statistics, seasonally adjusted)
us_cpi_monthly   = resample_to_month_end(fred.get_series('CPIAUCSL'))
us_cpi_quarterly = us_cpi_monthly.resample('QE').last()
us_cpi_yoy       = us_cpi_quarterly.pct_change(4) * 100
us_cpi_yoy.name  = 'us_cpi'

# US Unemployment Rate — monthly to quarterly
# Source: FRED UNRATE (Bureau of Labor Statistics, seasonally adjusted)
# Switched from OECD — Q4 2025 missing due to same publication lag issue
us_unemployment_monthly   = resample_to_month_end(fred.get_series('UNRATE'))
us_unemployment_quarterly = us_unemployment_monthly.resample('QE').last()
us_unemployment_quarterly.name = 'us_unemployment'
print(f'US Unemployment Q4 2025 (FRED UNRATE): {us_unemployment_quarterly.loc["2025-12-31"]:.4f}%')

# Compile US FRED DataFrame
fred_us = pd.DataFrame({
    'us_real_gdp':              us_real_gdp,
    'us_gdp_yoy_growth':        us_gdp_yoy_growth,
    'us_bond_yield_10y':        us_bond_yield_10y,
    'us_bond_yield_1y':         us_bond_yield_1y,
    'us_term_spread':           us_term_spread,
    'us_reer':                  us_reer,
    'us_reer_log_ret':          us_reer_log_ret,
    'us_oil_price':             us_oil_price,
    'us_oil_log_ret':           us_oil_log_ret,
    'us_credit':                us_credit,
    'us_credit_qoq_growth':     us_credit_qoq_growth,
    'us_credit_yoy_growth':     us_credit_yoy_growth,
    'us_industrial_production': us_industrial_production,
    'us_vix':                   us_vix,
    'us_vix_log_ret':           us_vix_log_ret,
    'us_delinquency_rate':      us_delinquency_rate,
    'us_cpi':                   us_cpi_yoy,
    'us_unemployment':          us_unemployment_quarterly,
})

fred_us.index = pd.to_datetime(fred_us.index).tz_localize(None)
print(f"US FRED data: {fred_us.index.min().date()} to {fred_us.index.max().date()}, {len(fred_us)} rows")

US Unemployment Q4 2025 (FRED UNRATE): 4.4000%
US FRED data: 1919-03-31 to 2026-06-30, 430 rows


In [32]:
fred_us.head()

,us_real_gdp,us_gdp_yoy_growth,us_bond_yield_10y,us_bond_yield_1y,us_term_spread,us_reer,us_reer_log_ret,us_oil_price,us_oil_log_ret,us_credit,us_credit_qoq_growth,us_credit_yoy_growth,us_industrial_production,us_vix,us_vix_log_ret,us_delinquency_rate,us_cpi,us_unemployment
1919-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5238,NaN,NaN,NaN,NaN,NaN
1919-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.9277,NaN,NaN,NaN,NaN,NaN
1919-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.1970,NaN,NaN,NaN,NaN,NaN
1919-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.1432,NaN,NaN,NaN,NaN,NaN
1920-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.5201,NaN,NaN,NaN,NaN,NaN


In [33]:
fred_us.tail()

,us_real_gdp,us_gdp_yoy_growth,us_bond_yield_10y,us_bond_yield_1y,us_term_spread,us_reer,us_reer_log_ret,us_oil_price,us_oil_log_ret,us_credit,us_credit_qoq_growth,us_credit_yoy_growth,us_industrial_production,us_vix,us_vix_log_ret,us_delinquency_rate,us_cpi,us_unemployment
2025-06-30,23770.976,2.080467,4.38,3.96,0.42,108.72,-0.037815,66.30,-0.080669,42224.199,0.695880,1.771276,101.4785,16.73,-0.286486,3.04,2.680454,4.1
2025-09-30,24026.834,2.335168,4.12,3.68,0.44,108.27,-0.004148,63.17,-0.048360,42637.245,0.978221,1.916437,101.6680,16.28,-0.027266,2.98,3.022572,4.4
2025-12-31,24055.749,1.989300,4.14,3.48,0.66,107.51,-0.007044,57.26,-0.098227,NaN,NaN,NaN,101.6781,14.95,-0.085226,2.94,2.653304,4.4
2026-03-31,NaN,NaN,4.13,3.68,0.45,105.88,-0.015277,102.86,0.585767,NaN,NaN,NaN,102.5510,25.25,0.524115,NaN,3.285958,4.3
2026-06-30,NaN,NaN,NaN,3.70,NaN,NaN,NaN,114.01,0.102917,NaN,NaN,NaN,NaN,19.12,-0.278091,NaN,NaN,NaN


## 5. Yahoo Finance — Equity Indices
---
Daily prices are fetched for the full available history and resampled to quarterly frequency by taking the closing price of the last trading day of each quarter. This is consistent with the quarter-end convention used across all other sources.

In [34]:
# UK: FTSE 100
ftse_raw           = yf.Ticker("^FTSE").history(start="1984-01-01", interval="1d")
ftse_close         = ftse_raw['Close'].resample('QE').last().rename('uk_ftse_close')
ftse_close.index   = ftse_close.index.tz_localize(None)
uk_ftse_qoq_growth = ftse_close.pct_change(1) * 100
uk_ftse_qoq_growth.name = 'uk_ftse_qoq_growth'

# US: S&P 500
sp500_raw           = yf.Ticker("^GSPC").history(start="1950-01-01", interval="1d")
sp500_close         = sp500_raw['Close'].resample('QE').last().rename('us_sp500_close')
sp500_close.index   = sp500_close.index.tz_localize(None)
us_sp500_log_ret    = np.log(sp500_close / sp500_close.shift(1)).rename('us_sp500_log_ret')

print(f"FTSE 100: {ftse_close.index.min().date()} to {ftse_close.index.max().date()}, {len(ftse_close)} quarters")
print(f"S&P 500:  {sp500_close.index.min().date()} to {sp500_close.index.max().date()}, {len(sp500_close)} quarters")

FTSE 100: 1984-03-31 to 2026-06-30, 170 quarters
S&P 500:  1950-03-31 to 2026-06-30, 306 quarters


## 6. OECD Data
---
Four series are fetched from the OECD SDMX REST API for both countries:

1. Real House Price Index — quarterly
2. Consumer Price Index YoY change — quarterly
3. Unemployment rate — quarterly
4. Consumer Confidence Index — monthly, resampled to quarterly for the quarterly dataset

A one-second delay is applied between each API call to respect OECD rate limits. The `fetch_oecd` helper handles date parsing and returns a clean two-column DataFrame for each series. The `build_oecd_dataframe` function fetches all series for a given country and joins them on the date index.

In [35]:
# OECD API query definitions: {column_name: url}
# Quarterly series use YYYY-Qn period-format dates
# Monthly series (consumer confidence) are resampled to quarterly after fetching

oecd_queries_uk = {
    'uk_house_price_idx':     "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,1.0/GBR.Q.RHP.IX?startPeriod=1968-Q1&endPeriod=2026-Q1&dimensionAtObservation=AllDimensions",
    'uk_cpi':                 "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_PRICES@DF_PRICES_ALL,1.0/GBR.Q.N.CPI.PA._T.N.GY?startPeriod=1914-Q1&endPeriod=2026-Q1&dimensionAtObservation=AllDimensions",
    'uk_unemployment':        "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_INDIC,1.0/GBR.UNE_LF_M...Y._T.Y_GE15..Q?startPeriod=1955-Q1&endPeriod=2026-Q4&dimensionAtObservation=AllDimensions",
    'uk_consumer_confidence': "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_CLI,/GBR.M.LI...AA...H?startPeriod=1955-01&endPeriod=2026-12&dimensionAtObservation=AllDimensions",
}

oecd_queries_us = {
    'us_house_price_idx':     "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,1.0/USA.Q.RHP.IX?startPeriod=1960-Q1&endPeriod=2026-Q1&dimensionAtObservation=AllDimensions",
#    'us_cpi':                 "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_PRICES@DF_PRICES_ALL,1.0/USA.Q.N.CPI.PA._T.N.GY?startPeriod=1914-Q1&endPeriod=2026-Q1&dimensionAtObservation=AllDimensions",
#    'us_unemployment':        "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_INDIC,1.0/USA.UNE_LF_M...Y._T.Y_GE15..Q?startPeriod=1955-Q1&endPeriod=2026-Q4&dimensionAtObservation=AllDimensions",
    'us_consumer_confidence': "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_CLI,/USA.M.LI...AA...H?startPeriod=1955-01&endPeriod=2026-12&dimensionAtObservation=AllDimensions",
}

print("OECD query URLs defined.")

OECD query URLs defined.


In [36]:
def build_oecd_dataframe(queries, monthly_cols=None):
    """
    Fetch all OECD series defined in queries, handle date parsing,
    and join into a single DataFrame indexed by quarter-end timestamps.

    monthly_cols: list of column names that are monthly (YYYY-MM format).
    These are resampled to quarter-end before joining.
    All other columns are assumed quarterly (YYYY-Qn format).
    """
    if monthly_cols is None:
        monthly_cols = []

    frames = []
    for col_name, url in queries.items():
        df = fetch_oecd(url, col_name)
        if df is None:
            continue

        if col_name in monthly_cols:
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date').resample('QE').last()
            df.index = df.index.normalize()
        else:
            df['date'] = pd.PeriodIndex(df['date'], freq='Q').to_timestamp(how='end')
            df = df.set_index('date')
            df.index = df.index.normalize()

        frames.append(df)

    result = pd.concat(frames, axis=1)
    result.index = pd.to_datetime(result.index).tz_localize(None)
    return result


# Fetch UK OECD data
oecd_uk = build_oecd_dataframe(
    oecd_queries_uk,
    monthly_cols=['uk_consumer_confidence']
)

# Fetch US OECD data
oecd_us = build_oecd_dataframe(
    oecd_queries_us,
    monthly_cols=['us_consumer_confidence']
)

print(f"OECD UK: {oecd_uk.index.min().date()} to {oecd_uk.index.max().date()}, {len(oecd_uk)} rows")
print(f"OECD US: {oecd_us.index.min().date()} to {oecd_us.index.max().date()}, {len(oecd_us)} rows")

OECD UK: 1956-03-31 to 2026-03-31, 281 rows
OECD US: 1955-03-31 to 2026-03-31, 285 rows


## 7. Combine into Final Datasets
---

### 7.a Final Quarterly Datasets

All sources are joined on the datetime index using an outer join to preserve the full history of each series. Some rows will have NaNs where a given series starts later or ends earlier than others. The current incomplete quarter is dropped to avoid including partial observations.

In [37]:
# Normalise all indices to midnight to ensure exact date alignment across sources
for df in [fred_uk, fred_us, oecd_uk, oecd_us]:
    df.index = pd.to_datetime(df.index).normalize()

ftse_close.index         = pd.to_datetime(ftse_close.index).normalize()
uk_ftse_qoq_growth.index = pd.to_datetime(uk_ftse_qoq_growth.index).normalize()
sp500_close.index        = pd.to_datetime(sp500_close.index).normalize()
us_sp500_log_ret.index   = pd.to_datetime(us_sp500_log_ret.index).normalize()

# UK quarterly master dataset
macro_uk = oecd_uk.join(
    [fred_uk, ftse_close, uk_ftse_qoq_growth],
    how='outer'
).sort_index()

# US quarterly master dataset
macro_us = oecd_us.join(
    [fred_us, sp500_close, us_sp500_log_ret],
    how='outer'
).sort_index()

# Drop current and any future incomplete quarters
# Use the last fully completed quarter (one quarter before today)
last_complete_quarter = pd.Timestamp('2025-12-31')
macro_uk = macro_uk[macro_uk.index <= last_complete_quarter]
macro_us = macro_us[macro_us.index <= last_complete_quarter]

print("UK quarterly dataset shape:", macro_uk.shape)
print("US quarterly dataset shape:", macro_us.shape)

UK quarterly dataset shape: (424, 15)
US quarterly dataset shape: (428, 22)


### 7b. US Monthly Dataset

The monthly dataset serves Stage 1 of the pipeline — macro variable forecasting via SARIMA, SARIMAX, and ML models. Monthly frequency provides more observations for time series model fitting compared to the quarterly dataset.

**Strategy:**

Variables with native monthly or daily frequency are fetched at source frequency and resampled to month-end using the last observation of the month. Quarterly-only variables are disaggregated to monthly using cubic spline interpolation (`scipy.interpolate.CubicSpline`), which preserves the original quarterly values exactly at quarter-end anchor dates and smoothly fills the two intervening months.

Log returns are computed at monthly frequency for the same financial price variables as in the quarterly dataset (VIX, oil, S&P 500, REER). Growth rates for GDP and credit are recomputed directly on the monthly level series rather than interpolating the growth rates themselves, which is the methodologically correct approach.

The monthly dataset contains the same target variable (`us_delinquency_rate`) disaggregated via cubic spline interpolation, making it available for monthly frequency model experiments if needed.

In [38]:
# Step 1: Native monthly and daily pulls from FRED
# All series resampled to month-end using last observation of the month

us_bond_yield_10y_m = resample_to_month_end(
    fred.get_series('IRLTLT01USM156N')
).rename('us_bond_yield_10y')

us_bond_yield_1y_m = resample_to_month_end(
    fred.get_series('DGS1')
).rename('us_bond_yield_1y')

us_reer_m = resample_to_month_end(
    fred.get_series('RBUSBIS')
).rename('us_reer')

us_oil_price_m = resample_to_month_end(
    fred.get_series('DCOILWTICO')
).rename('us_oil_price')

us_industrial_production_m = resample_to_month_end(
    fred.get_series('INDPRO')
).rename('us_industrial_production')

us_vix_m = resample_to_month_end(
    fred.get_series('VIXCLS')
).rename('us_vix')

# US CPI — native monthly, YoY growth rate
# Source: FRED CPIAUCSL (Bureau of Labor Statistics, seasonally adjusted)
us_cpi_m = resample_to_month_end(
    fred.get_series('CPIAUCSL')
).pct_change(12).mul(100).rename('us_cpi')

# US Unemployment Rate — native monthly
# Source: FRED UNRATE (Bureau of Labor Statistics, seasonally adjusted)
us_unemployment_m = resample_to_month_end(
    fred.get_series('UNRATE')
).rename('us_unemployment')

# Term spread computed from monthly yields
us_term_spread_m = (us_bond_yield_10y_m - us_bond_yield_1y_m).rename('us_term_spread')

print("Native monthly FRED series fetched.")
print(f"  Bond yield 10Y: {us_bond_yield_10y_m.index.min().date()} to {us_bond_yield_10y_m.index.max().date()}")
print(f"  VIX:            {us_vix_m.index.min().date()} to {us_vix_m.index.max().date()}")
print(f"  CPI YoY:        {us_cpi_m.index.min().date()} to {us_cpi_m.index.max().date()}")

C:\Users\andre\AppData\Local\Temp\ipykernel_25212\837735418.py:32: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ).pct_change(12).mul(100).rename('us_cpi')


Native monthly FRED series fetched.
  Bond yield 10Y: 1953-04-30 to 2026-02-28
  VIX:            1990-01-31 to 2026-04-30
  CPI YoY:        1947-01-31 to 2026-03-31


In [39]:
# Step 2: S&P 500 at native daily frequency, resampled to month-end

sp500_raw_m      = yf.Ticker("^GSPC").history(start="1950-01-01", interval="1d")
us_sp500_m       = sp500_raw_m['Close'].resample('ME').last()
us_sp500_m.index = us_sp500_m.index.tz_localize(None)
us_sp500_m.name  = 'us_sp500_close'

print(f"S&P 500 monthly: {us_sp500_m.index.min().date()} to {us_sp500_m.index.max().date()}, {len(us_sp500_m)} months")

S&P 500 monthly: 1950-01-31 to 2026-04-30, 916 months


In [40]:
# Step 3: Consumer confidence at native monthly frequency
# Uses fetch_oecd_native_monthly to avoid resampling to quarterly

us_consumer_confidence_m = fetch_oecd_native_monthly(
    "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_CLI,/USA.M.LI...AA...H"
    "?startPeriod=1955-01&endPeriod=2026-12&dimensionAtObservation=AllDimensions",
    'us_consumer_confidence'
)

print(f"Consumer confidence monthly: {us_consumer_confidence_m.index.min().date()} "
      f"to {us_consumer_confidence_m.index.max().date()}, {len(us_consumer_confidence_m)} months")

Consumer confidence monthly: 1955-01-31 to 2026-03-31, 855 months


In [41]:
# Step 4: Quarterly to monthly via cubic spline interpolation
# Growth rates for GDP and credit are recomputed on the monthly level series
# rather than interpolating the growth rates directly

# GDP
us_real_gdp_m       = quarterly_to_monthly_spline(us_real_gdp_raw.rename('us_real_gdp'))
us_gdp_yoy_growth_m = us_real_gdp_m.pct_change(12).mul(100).rename('us_gdp_yoy_growth')

# Credit
us_credit_m            = quarterly_to_monthly_spline(us_credit_raw.rename('us_credit'))
us_credit_mom_growth   = us_credit_m.pct_change(1).mul(100).rename('us_credit_mom_growth')
us_credit_yoy_growth_m = us_credit_m.pct_change(12).mul(100).rename('us_credit_yoy_growth')

# House prices
us_house_price_m = quarterly_to_monthly_spline(
    oecd_us['us_house_price_idx'].rename('us_house_price_idx')
)

# CPI
#us_cpi_m = quarterly_to_monthly_spline(
#    oecd_us['us_cpi'].rename('us_cpi')
#)

# Unemployment
#us_unemployment_m = quarterly_to_monthly_spline(
#    oecd_us['us_unemployment'].rename('us_unemployment')
#)

# Delinquency rate — target variable
us_delinquency_m = quarterly_to_monthly_spline(
    us_delinquency_raw.rename('us_delinquency_rate')
)

print("Cubic spline interpolation complete.")

Cubic spline interpolation complete.


In [42]:
# Step 5: Log returns for financial price variables at monthly frequency
# Formula: ln(P_t / P_{t-1})

us_vix_log_ret   = np.log(us_vix_m / us_vix_m.shift(1)).rename('us_vix_log_ret')
us_oil_log_ret   = np.log(us_oil_price_m / us_oil_price_m.shift(1)).rename('us_oil_log_ret')
us_sp500_log_ret = np.log(us_sp500_m / us_sp500_m.shift(1)).rename('us_sp500_log_ret')
us_reer_log_ret  = np.log(us_reer_m / us_reer_m.shift(1)).rename('us_reer_log_ret')

print("Log returns computed for VIX, oil, S&P 500, and REER.")

Log returns computed for VIX, oil, S&P 500, and REER.


In [43]:
# Step 6: Compile US monthly dataset

monthly_series = [
    us_real_gdp_m,
    us_gdp_yoy_growth_m,
    us_bond_yield_10y_m,
    us_bond_yield_1y_m,
    us_term_spread_m,
    us_reer_m,
    us_reer_log_ret,
    us_oil_price_m,
    us_oil_log_ret,
    us_credit_m,
    us_credit_mom_growth,
    us_credit_yoy_growth_m,
    us_industrial_production_m,
    us_vix_m,
    us_vix_log_ret,
    us_sp500_m,
    us_sp500_log_ret,
    us_house_price_m,
    us_cpi_m,
    us_unemployment_m,
    us_consumer_confidence_m,
    us_delinquency_m,
]

macro_us_monthly = pd.concat(monthly_series, axis=1).sort_index()

# Drop the current incomplete month
current_month_end = pd.Timestamp.today().to_period('M').to_timestamp('M')
macro_us_monthly = macro_us_monthly[macro_us_monthly.index < current_month_end]

print("US monthly dataset shape:", macro_us_monthly.shape)
print(f"Coverage: {macro_us_monthly.index.min().date()} to {macro_us_monthly.index.max().date()}")

US monthly dataset shape: (1287, 22)
Coverage: 1919-01-31 to 2026-03-31


## 8. Coverage Audit
---
Check the start date, end date, and number of non-null observations for each variable across all three output datasets. This gives a clear picture of which series have limited historical coverage and what the effective analytical window is for each dataset.

In [44]:
def coverage_summary(df, label):
    """
    Print a table showing for each column: first non-null date,
    last non-null date, count of non-null observations, and count
    of missing observations.
    """
    print(f"\n{'='*60}")
    print(f"  Coverage Summary: {label}")
    print(f"{'='*60}")
    summary = pd.DataFrame({
        'First Obs':  df.apply(lambda c: c.first_valid_index()),
        'Last Obs':   df.apply(lambda c: c.last_valid_index()),
        'N Non-Null': df.notna().sum(),
        'N Missing':  df.isna().sum(),
    })
    print(summary.to_string())

    complete_rows = df.dropna().shape[0]
    print(f"\n  Complete rows (no missing values): {complete_rows} of {len(df)}")


coverage_summary(macro_us,         "US Macroeconomic Variables — Quarterly")
coverage_summary(macro_us_monthly, "US Macroeconomic Variables — Monthly")
coverage_summary(macro_uk,         "UK Macroeconomic Variables — Quarterly")


  Coverage Summary: US Macroeconomic Variables — Quarterly
                          First Obs   Last Obs  N Non-Null  N Missing
us_house_price_idx       1970-03-31 2025-12-31         224        204
us_consumer_confidence   1955-03-31 2025-12-31         284        144
us_real_gdp              1947-03-31 2025-12-31         316        112
us_gdp_yoy_growth        1948-03-31 2025-12-31         312        116
us_bond_yield_10y        1953-06-30 2025-12-31         291        137
us_bond_yield_1y         1962-03-31 2025-12-31         256        172
us_term_spread           1962-03-31 2025-12-31         256        172
us_reer                  1994-03-31 2025-12-31         128        300
us_reer_log_ret          1994-06-30 2025-12-31         127        301
us_oil_price             1986-03-31 2025-12-31         160        268
us_oil_log_ret           1986-06-30 2025-12-31         159        269
us_credit                1945-12-31 2025-09-30         320        108
us_credit_qoq_growth     1946-

## 9. Save to CSV
---
Three output files are saved to the specified output directory. File naming follows a consistent convention — country code followed by frequency suffix (`_Q` for quarterly, `_M` for monthly).

| File | Description |
|---|---|
| `MacroVariables_US_Q.csv` | US quarterly — primary modelling dataset including target variable |
| `MacroVariables_US_M.csv` | US monthly — Stage 1 macro forecasting dataset |
| `MacroVariables_UK_Q.csv` | UK quarterly — macro variables only, no target variable |

In [45]:
from pathlib import Path

# Saves to Data Collection/ relative to this notebook
# ST-498/
# ├── Code/             ← this notebook lives here
# └── Data Collection/  ← outputs saved here
OUTPUT_DIR = Path.cwd().parent / 'Data Collection'
OUTPUT_DIR.mkdir(exist_ok=True)

macro_us.to_csv(OUTPUT_DIR / 'MacroVariables_US_Q.csv')
macro_us_monthly.to_csv(OUTPUT_DIR / 'MacroVariables_US_M.csv')
macro_uk.to_csv(OUTPUT_DIR / 'MacroVariables_UK_Q.csv')

print(f'Saved to: {OUTPUT_DIR}')
print(f'  MacroVariables_US_Q.csv   {macro_us.shape[0]} rows x {macro_us.shape[1]} cols')
print(f'  MacroVariables_US_M.csv   {macro_us_monthly.shape[0]} rows x {macro_us_monthly.shape[1]} cols')
print(f'  MacroVariables_UK_Q.csv   {macro_uk.shape[0]} rows x {macro_uk.shape[1]} cols')

Saved to: c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection
  MacroVariables_US_Q.csv   428 rows x 22 cols
  MacroVariables_US_M.csv   1287 rows x 22 cols
  MacroVariables_UK_Q.csv   424 rows x 15 cols
